In [1]:
import os
import joblib
import pandas as pd
import xgboost as xgb
import warnings

warnings.filterwarnings('ignore')

MODEL_PATH = 'modelo_xgboost_final.pkl' 
SCALER_PATH = 'scaler_gpa.pkl'
QUANTILE_SCALER_PATH = 'quantile_scaler_gpa.pkl'

print("--- SISTEMA DE PREDICCIÓN DE GPA (XGBOOST OPTIMIZADO) ---")
print("Cargando componentes...")

try:
    model = joblib.load(MODEL_PATH) 
    scaler = joblib.load(SCALER_PATH)
    quantile_scaler = joblib.load(QUANTILE_SCALER_PATH)
    print("[OK] Modelo XGBoost y escaladores cargados correctamente.")
except Exception as e:
    print(f"[ERROR] Problema al cargar los archivos: {e}")
    exit()

print("-" * 55)

def obtener_input_numerico(prompt, tipo=float):
    while True:
        try:
            valor = input(prompt)
            return tipo(valor)
        except ValueError:
            print(f"[!] Entrada inválida. Se requiere un valor tipo {tipo.__name__}.")

def obtener_input_categorico(prompt, opciones_validas):
    while True:
        valor = input(prompt).strip().lower()
        if valor in opciones_validas:
            return valor
        print(f"[!] Opción inválida. Elige una de las siguientes: {opciones_validas}")

def iniciar_prediccion():
    print("\nPor favor, ingresa los datos del estudiante:\n")
    
    age = obtener_input_numerico("1. Edad del adolescente (ej. 16): ", int)
    social_media = obtener_input_numerico("2. Horas diarias en redes sociales (ej. 3.5): ", float)
    sleep = obtener_input_numerico("3. Horas de sueño (ej. 7.5): ", float)
    screen_sleep = obtener_input_numerico("4. Horas de pantalla antes de dormir (ej. 1.0): ", float)
    physical = obtener_input_numerico("5. Horas de actividad física (ej. 1.0): ", float)
    stress = obtener_input_numerico("6. Nivel de estrés (numérico, ej. 3): ", float)
    anxiety = obtener_input_numerico("7. Nivel de ansiedad (numérico, ej. 2): ", float)
    addiction = obtener_input_numerico("8. Nivel de adicción (numérico, ej. 1): ", float)
    
    gender = obtener_input_categorico("9. Género (male / female): ", ['male', 'female'])
    platform = obtener_input_categorico("10. Plataforma principal (instagram / tiktok / both): ", ['instagram', 'tiktok', 'both'])
    social_int = obtener_input_categorico("11. Nivel de interacción social (low / medium / high): ", ['low', 'medium', 'high'])

    print("\nProcesando la Ingeniería de Características...")

    map_social = {'low': 0, 'medium': 1, 'high': 2}
    social_interaction_level = float(map_social[social_int])

    total_digital_load = social_media + screen_sleep
    sleep_quality_ratio = sleep / (screen_sleep + 1.0)
    mental_health_risk = stress + anxiety + addiction
    lifestyle_balance = (physical + social_interaction_level) / (social_media + 1.0)

    gender_male = 1.0 if gender == 'male' else 0.0
    platform_usage_Instagram = 1.0 if platform == 'instagram' else 0.0
    platform_usage_TikTok = 1.0 if platform == 'tiktok' else 0.0

    columnas_ordenadas = [
        'age', 'daily_social_media_hours', 'sleep_hours', 'screen_time_before_sleep', 
        'physical_activity', 'social_interaction_level', 'stress_level', 
        'anxiety_level', 'addiction_level', 
        'total_digital_load', 'sleep_quality_ratio', 'mental_health_risk', 'lifestyle_balance',
        'gender_male', 'platform_usage_Instagram', 'platform_usage_TikTok'
    ]
    
    datos_crudos = [[
        age, social_media, sleep, screen_sleep, physical, 
        social_interaction_level, stress, anxiety, addiction, 
        total_digital_load, sleep_quality_ratio, mental_health_risk, lifestyle_balance,
        gender_male, platform_usage_Instagram, platform_usage_TikTok
    ]]
    
    df_input = pd.DataFrame(datos_crudos, columns=columnas_ordenadas)

    try:
        col_standard = ['age', 'social_interaction_level', 'stress_level', 'anxiety_level', 'addiction_level']
        col_quantile = ['daily_social_media_hours', 'sleep_hours', 'screen_time_before_sleep', 'physical_activity', 'total_digital_load', 'sleep_quality_ratio', 'mental_health_risk', 'lifestyle_balance']
        
        df_input[col_standard] = scaler.transform(df_input[col_standard])
        df_input[col_quantile] = quantile_scaler.transform(df_input[col_quantile])
        
        gpa_predicho = model.predict(df_input)[0]
        
        gpa_final = max(0.0, min(4.0, float(gpa_predicho)))
        
        print("=" * 40)
        print("        RESULTADO DE LA PREDICCIÓN")
        print("=" * 40)
        print(f"  > Rendimiento Académico (GPA): {gpa_final:.3f}")
        print("=" * 40 + "\n")
        
    except Exception as e:
        print(f"\n[ERROR] Ocurrió un problema en la transformación o predicción: {e}")

if __name__ == "__main__":
    iniciar_prediccion()

--- SISTEMA DE PREDICCIÓN DE GPA (XGBOOST OPTIMIZADO) ---
Cargando componentes...
[OK] Modelo XGBoost y escaladores cargados correctamente.
-------------------------------------------------------

Por favor, ingresa los datos del estudiante:


Procesando la Ingeniería de Características...
        RESULTADO DE LA PREDICCIÓN
  > Rendimiento Académico (GPA): 2.985

